# Lilly — the outside comparison (NLLB-200 against Lilly, same FLORES pairs)

**Job:** `outside-baseline`. Pre-registered in `training/PREREGISTRATION.md`,
"v3 — an outside comparison, written before any outside number exists".

**What this run does.** Scores `facebook/nllb-200-distilled-600M` on the same
2,009 FLORES-200 pairs Lilly is scored on, in both directions, through
`training/evaluate.py`'s own loader, batching and sacrebleu call. Then scores
Lilly's two *bases* on the same GPU, in the same process order, as a
same-hardware anchor: if the bases reproduce their published CPU numbers here,
NLLB's numbers on this box are comparable to Lilly's published ones.

**What this run does NOT do.** It trains nothing, it ships no weights, and it
does not touch `models/lilly/`. The artefact is a small zip of JSON and a
markdown table. There is no adapter to install afterwards.

**Attach:** nothing. Internet must be **On** (Hugging Face for NLLB and the two
bases, `dl.fbaipublicfiles.com` for FLORES, GitHub for the code).

**Accelerator:** GPU. The notebook refuses to run without one.

In [ ]:
# 1. Stop here unless the machine is actually set up
# Kaggle marks a version COMPLETE whenever no cell RAISES — a shell command that
# fails is not enough. Every check below is Python and every later step is a
# checked subprocess whose output is teed into Output, so a run that produced no
# numbers cannot be mistaken for one that did.
import json, os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
GPU = torch.cuda.get_device_name(0)
print(torch.cuda.device_count(), "GPU(s) visible, using:", GPU)

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass          # a status code still proves we got out
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://huggingface.co",
             "https://dl.fbaipublicfiles.com"):
    reachable(host)
print("network ok")

# The child's stdout is NOT the Kaggle log. subprocess.run(check=True) can
# COMPLETE a kernel that never printed a single BLEU line, which is how a run
# gets judged on a number nobody can see. Tee everything into Output.
TEE = Path("/kaggle/working/stdout.txt")
TEE.parent.mkdir(parents=True, exist_ok=True)

def run(*cmd, quiet=False):
    line = "$ " + " ".join(str(c) for c in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "\n")
        child = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
        for out in child.stdout:
            if not quiet:
                print(out, end="", flush=True)
            sink.write(out)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)

In [ ]:
# 2. Get the Lilly code — into scratch, never into Output
# Everything under /kaggle/working becomes the version Output and `kaggle kernels
# output` walks that whole tree. A git clone there once buried the artefact under
# thousands of files. The clone goes to /kaggle/temp; only the result zip and the
# offload log belong in /kaggle/working.
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training").is_dir(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd())

sys.path.insert(0, str(CLONE / "training"))
from kaggle_offload import Offload
OFF = Offload("outside-baseline", os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "manual"))
OFF.hardware(GPU)
print("offload log started:", OFF.body["git"])

In [ ]:
# 3. Install what we need (~2 min)
# Versions come out of the repo's own requirements.txt rather than a second
# hand-kept list here. A notebook-local copy is how a speech run died: peft was
# pinned in requirements.txt, missing from the notebook, and Kaggle's much newer
# peft raised on the first call — after a 3 GB download.
NEEDED = ["transformers", "sacrebleu", "sentencepiece", "sacremoses",
          "huggingface_hub"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
print("pinned here:", [pins[n] for n in NEEDED if n in pins])
print("no pin, taking latest:", [n for n in NEEDED if n not in pins] or "none")
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])

In [ ]:
# 4. The data, and a hard count on it
# FLORES is the whole comparison. A run that scores a partial download is not the
# pre-registered comparison, so the count is asserted before anything is measured.
run(sys.executable, "data/scripts/download_flores.py")
sizes = {p.name: sum(1 for _ in p.open(encoding="utf-8"))
         for p in sorted((CLONE / "data" / "flores").glob("*.??"))}
print(sizes)
pairs = sizes.get("devtest.bs", 0) + sizes.get("dev.bs", 0)
if pairs != 2009:
    raise SystemExit(f"FLORES came back as {pairs} pairs, not 2009 — "
                     "the pre-registered comparison is all 2,009 and this is not it")
OFF.metric("flores_pairs", pairs, stage="data")
print("FLORES ok:", pairs, "pairs")

In [ ]:
# 5. The outside system's weights (~2.4 GB)
from huggingface_hub import snapshot_download
dest = CLONE / "models" / "external" / "nllb-600M"
dest.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id="facebook/nllb-200-distilled-600M", repo_type="model",
                  local_dir=str(dest),
                  allow_patterns=["*.json", "*.model", "*.bin", "*.txt"])
weights = dest / "pytorch_model.bin"
if not weights.is_file() or weights.stat().st_size < 1_000_000_000:
    raise SystemExit(f"NLLB weights missing or truncated at {weights}")
print(f"NLLB ready: {weights.stat().st_size / 1e9:.2f} GB")

In [ ]:
# 6. Smoke: eight pairs through the whole path, before the real run
# Loads the model, resolves the forced target token, decodes, scores. If the
# language code is wrong or the tokenizer cannot find it, this dies in two
# minutes instead of after both full directions.
run(sys.executable, "training/outside_baseline.py", "--direction", "bs-en",
    "--limit", "8", "--batch-size", "8")
smoke = json.loads((CLONE / "training" / "outside" / "nllb-600M-bs-en.json").read_text())
assert smoke["limited"] and smoke["pairs"] == 8, smoke
assert smoke["hyps"] and all(h.strip() for h in smoke["hyps"]), "empty translations"
print("smoke ok — the path works; the numbers above decide nothing")

In [ ]:
# 7. THE COMPARISON — NLLB-200 on all 2,009 pairs, both directions
# batch-size 16 on purpose: it is what the CPU cross-check used, and padding
# changes what a model produces (batching in file order instead of by length
# costs 8.7 BLEU on this set, measured). Same setting both sides, so a gap
# between the two runs would be hardware and not configuration.
for direction in ("bs-en", "en-bs"):
    run(sys.executable, "training/outside_baseline.py",
        "--direction", direction, "--batch-size", "16")

In [ ]:
# 8. The same-hardware anchor: Lilly's two bases, this GPU, this code
# Without this the comparison is NLLB-on-a-T4 against Lilly-on-a-CPU, and the
# speech lane already recorded a full point of word error between the two kinds
# of machine on identical clips. If these bases reproduce their published
# numbers here, NLLB's numbers on this box are comparable to Lilly's published
# ones; if they do not, that gap is the first thing the write-up has to explain.
# fetch_translate_base.py, not fetch_models.py: the second one pulls the
# PUBLISHED ctranslate2 bundle, and its own comment says it deliberately skips
# "translate" -- the untuned float32 base that only training and this scorer
# read. Each direction's base is a different upstream model.
for direction in ("bs-en", "en-bs"):
    run(sys.executable, "scripts/fetch_translate_base.py", "--direction", direction)
for direction in ("bs-en", "en-bs"):
    run(sys.executable, "training/verify_base_flores.py", "--direction", direction)

In [ ]:
# 9. The gate, then the package. Nothing is zipped before this cell passes.
OFF.check_trainproof(TEE)

RES = CLONE / "training" / "outside"
FR = CLONE / "training" / "form-rate"
rows, missing = [], []
for direction in ("bs-en", "en-bs"):
    nllb_path = RES / f"nllb-600M-{direction}.json"
    base_path = FR / f"base-flores-{direction}.json"
    for p in (nllb_path, base_path):
        if not p.is_file():
            missing.append(str(p.relative_to(CLONE)))
if missing:
    raise SystemExit("these results were never written: " + ", ".join(missing))

report = ["# NLLB-200-distilled-600M against Lilly — FLORES-200, 2,009 pairs",
          "", f"Kaggle, {GPU}. Generated by the outside-baseline notebook.", ""]
for direction in ("bs-en", "en-bs"):
    n = json.loads((RES / f"nllb-600M-{direction}.json").read_text())
    b = json.loads((FR / f"base-flores-{direction}.json").read_text())
    # A limited run is a smoke test, not the comparison. Refuse to package one.
    if n["limited"] or n["pairs"] != 2009 or len(n["hyps"]) != 2009:
        raise SystemExit(f"{direction}: NLLB result is not the full 2,009 "
                         f"(limited={n['limited']}, pairs={n['pairs']}, "
                         f"hyps={len(n['hyps'])})")
    if b["pairs"] != 2009:
        raise SystemExit(f"{direction}: base anchor scored {b['pairs']} pairs, not 2009")
    empty = sum(1 for h in n["hyps"] if not h.strip())
    if empty:
        raise SystemExit(f"{direction}: {empty} empty translations — not a scoreable run")
    OFF.metric(f"nllb_bleu_{direction}", n["bleu"], stage="outside")
    OFF.metric(f"nllb_chrf2_{direction}", n["chrf2"], stage="outside")
    OFF.metric(f"base_bleu_{direction}", b["bleu"], stage="anchor")
    OFF.metric(f"base_chrf2_{direction}", b["chrf2"], stage="anchor")
    drift = b["bleu"] - b["committed"]["bleu"]
    rows += [f"## {direction}", "",
             "| system | params | BLEU | chrF2 |", "|---|---|---|---|",
             f"| NLLB-200-distilled-600M | 600M | {n['bleu']:.2f} | {n['chrf2']:.2f} |",
             f"| Lilly base, this GPU | {n['lilly']['params']} | {b['bleu']:.2f} | {b['chrf2']:.2f} |",
             f"| Lilly base, published (CPU) | | {b['committed']['bleu']:.2f} | {b['committed']['chrf2']:.2f} |",
             f"| Lilly shipped, published (CPU) | | {n['lilly']['bleu']:.2f} | {n['lilly']['chrf2']:.2f} |",
             "",
             f"Anchor drift, this GPU against the published CPU number: "
             f"**{drift:+.2f} BLEU**. Read every NLLB-vs-Lilly gap below with "
             f"this in front of it.", ""]
    print("\n".join(rows[-9:]))
report += rows
(CLONE / "training" / "RESULTS-outside-baseline.md").write_text(
    "\n".join(report) + "\n", encoding="utf-8")

import shutil, zipfile
ZIP = Path("/kaggle/working/lilly-outside-baseline.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RES.glob("*.json")):
        z.write(p, f"outside/{p.name}")
    for direction in ("bs-en", "en-bs"):
        p = FR / f"base-flores-{direction}.json"
        z.write(p, f"form-rate/{p.name}")
    z.write(CLONE / "training" / "RESULTS-outside-baseline.md",
            "RESULTS-outside-baseline.md")
size = ZIP.stat().st_size
if size < 10_000:
    raise SystemExit(f"zip is {size} bytes — that is not a result")
OFF.finish("complete", [ZIP.name])

out = list(Path("/kaggle/working").rglob("*"))
print(f"\nOutput holds {len(out)} entries, zip {size/1024:.0f} KB")
assert len(out) < 50, [str(p) for p in out[:50]]
print("packaged", ZIP.name)

**Status ERROR or CANCEL → do not use these numbers.** Recovery is not success:
a zip recovered from a cancelled run is not this run's artefact. Read
`stdout.txt` in Output, fix the cause, and relaunch — do not soften a gate so
the next version reaches COMPLETE past the same hole.

**On COMPLETE:** fetch with `scripts/kaggle_train.py outside-baseline --fetch`,
unzip beside `training/`, commit `training/outside/`,
`training/form-rate/base-flores-*.json` and `RESULTS-outside-baseline.md`, and
write the result up **whichever way it fell** — that is fixed in the
pre-registration and is not reopened once the numbers exist.